# 📷 Real-Time Sign Language Detection
ใช้ `best_model.keras` จาก baselineCNN พร้อม OpenCV เพื่อ detect ภาษามือแบบ real-time ผ่านกล้อง

**วิธีใช้:**
1. Run ทุก cell ตามลำดับ
2. Cell สุดท้ายจะเปิดหน้าต่างกล้อง
3. วางมือในกรอบสีเขียว
4. กด `q` เพื่อปิด

In [ ]:
# ── 0. Install dependencies (ถ้ายังไม่มี) ─────────────────
# !pip install opencv-python tensorflow numpy

In [1]:
# ── 1. Imports ─────────────────────────────────────────────
import cv2
import numpy as np
import tensorflow as tf
import time

print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")

TensorFlow : 2.21.0
OpenCV     : 4.13.0


In [6]:
# ── 2. Config ──────────────────────────────────────────────
MODEL_PATH  = "outputs/best_model.keras"   # path ของโมเดล
IMG_SIZE    = 28                            # ขนาดภาพที่โมเดล expect
CONFIDENCE  = 0.60                         # threshold ขั้นต่ำในการแสดงผล

# Sign Language MNIST ไม่มี J (9) และ Z (25)
CLASS_NAMES = [c for c in 'ABCDEFGHIKLMNOPQRSTUVWXY']  # 24 classes
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

Classes (24): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


In [7]:
# ── 3. Load Model ──────────────────────────────────────────
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
print("\n✅ Model loaded successfully!")

Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_1 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 28, 28, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_1 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 14, 14, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_1 (BatchNormalization)      │ (None, 7, 7, 128)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_fc (BatchNormalization)      │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_fc (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 24)             │         6,168 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,325,162 (5.06 MB)

 Trainable params: 441,336 (1.68 MB)

 Non-trainable params: 1,152 (4.50 KB)

 Optimizer params: 882,674 (3.37 MB)


✅ Model loaded successfully!


In [8]:
# ── 4. Preprocessing Function ──────────────────────────────
def preprocess_roi(roi):
    # 1. Grayscale (เหมือนเดิม)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # 2. ✨ CLAHE — ปรับ contrast แบบ local (ช่วยมากเมื่อแสงไม่สม่ำเสมอ)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)

    # 3. ✨ Gaussian blur เบาๆ — ลด noise จากกล้อง
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # 4. Resize 28×28 (เหมือนเดิม)
    resized = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)

    # 5. ✨ Normalize แบบ per-image (ดีกว่า /255 เมื่อ brightness ต่างกัน)
    resized = resized.astype(np.float32)
    mean, std = resized.mean(), resized.std()
    normalized = (resized - mean) / (std + 1e-6)
    # clip ให้อยู่ใน range [0, 1] เพื่อให้ตรงกับ training
    normalized = np.clip((normalized + 3) / 6, 0.0, 1.0)

    return normalized.reshape(1, 28, 28, 1)

def predict(roi):
    """
    ทำนายจาก ROI
    Returns: (label, confidence)
    """
    img = preprocess_roi(roi)
    probs = model.predict(img, verbose=0)[0]  # shape (24,)
    idx   = int(np.argmax(probs))
    conf  = float(probs[idx])
    return CLASS_NAMES[idx], conf


print("✅ Functions ready")

✅ Functions ready


In [9]:
# ── 5. Real-Time Detection Loop ────────────────────────────
# กดปุ่ม  q  เพื่อออก

# ─── กรอบ ROI (กล่องสีเขียว) ───
BOX_X, BOX_Y = 100, 100    # มุมบนซ้าย
BOX_SIZE     = 300          # ขนาดกล่อง (pixel)

cap = cv2.VideoCapture(0)   # 0 = default webcam

if not cap.isOpened():
    raise RuntimeError("❌ ไม่สามารถเปิดกล้องได้ ลอง VideoCapture(1) หรือ VideoCapture(2)")

# ── FPS counter ──
prev_time = time.time()

# ── Prediction smoothing (เก็บ N predictions ล่าสุด) ──
from collections import deque
SMOOTH_N   = 5
pred_queue = deque(maxlen=SMOOTH_N)

print("📷 กล้องเปิดแล้ว — วางมือในกรอบสีเขียว | กด 'q' เพื่อออก")

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ อ่าน frame ไม่ได้")
        break

    # Flip แนวนอนให้เหมือนกระจก
    frame = cv2.flip(frame, 1)

    # ── ตัด ROI ──
    x1, y1 = BOX_X, BOX_Y
    x2, y2 = BOX_X + BOX_SIZE, BOX_Y + BOX_SIZE
    roi = frame[y1:y2, x1:x2]

    # ── Predict ──
    label, conf = predict(roi)

    # ── Smoothing: เฉพาะ prediction ที่มั่นใจ ──
    if conf >= CONFIDENCE:
        pred_queue.append(label)

    # หา label ที่ถูก vote มากสุด
    if pred_queue:
        from collections import Counter
        smooth_label = Counter(pred_queue).most_common(1)[0][0]
    else:
        smooth_label = "?"

    # ── วาด UI ──

    # กล่อง ROI
    box_color = (0, 255, 0) if conf >= CONFIDENCE else (0, 165, 255)
    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

    # แสดงผลการทำนาย
    text_label = f"{smooth_label}  ({conf*100:.1f}%)"
    cv2.putText(frame, text_label,
                (x1, y1 - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2,
                box_color, 3, cv2.LINE_AA)

    # FPS
    now      = time.time()
    fps      = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now
    cv2.putText(frame, f"FPS: {fps:.1f}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (200, 200, 200), 2, cv2.LINE_AA)

    # คำแนะนำ
    cv2.putText(frame, "Press 'q' to quit",
                (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (180, 180, 180), 1, cv2.LINE_AA)

    # แสดง preview ของ ROI ที่ถูก preprocess (มุมขวาล่าง)
    roi_preview = cv2.resize(
        cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY),
        (112, 112)
    )
    roi_bgr = cv2.cvtColor(roi_preview, cv2.COLOR_GRAY2BGR)
    ph, pw  = frame.shape[:2]
    frame[ph-120:ph-8, pw-120:pw-8] = roi_bgr
    cv2.rectangle(frame, (pw-121, ph-121), (pw-7, ph-7), (100,100,100), 1)

    cv2.imshow("Sign Language Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("\n✅ ปิดกล้องแล้ว")

📷 กล้องเปิดแล้ว — วางมือในกรอบสีเขียว | กด 'q' เพื่อออก

✅ ปิดกล้องแล้ว


## 💡 Tips สำหรับผลลัพธ์ที่ดีขึ้น

| ปัญหา | แนวทางแก้ |
|---|---|
| ทำนายผิดบ่อย | แสงให้สม่ำเสมอ, พื้นหลังเรียบ |
| กล้องเปิดไม่ได้ | เปลี่ยน `VideoCapture(0)` → `(1)` หรือ `(2)` |
| confidence ต่ำ | ปรับ `CONFIDENCE` ลงเหลือ `0.4` |
| กล่องเล็กเกินไป | เพิ่ม `BOX_SIZE` เช่น `400` |
| Jupyter ไม่เปิดหน้าต่าง | ใช้ script `.py` แทน หรือใช้ `%matplotlib inline` |

### ใช้เป็น Script แทน Jupyter
```bash
jupyter nbconvert --to script camera_detect.ipynb
python camera_detect.py
```